# RedditPages — sanity-check explorer

Loads all three tables into pandas DataFrames at startup: `users`, `posts`, `comments`. Use pandas idioms (`.query()`, `.groupby()`, etc.) for everything downstream — no SQL needed.

Memory: ~350 MB for the current `important.db` (56 users, 188K posts, 568K comments). Loading takes ~5s.

Set `target_user` once below and every per-user cell will use it.

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

DB_PATH = Path('../important.db').resolve()
target_user = 'LeaderAtLeading'

con = sqlite3.connect(f'file:{DB_PATH}?mode=ro', uri=True)
users    = pd.read_sql_query('SELECT * FROM user',    con)
posts    = pd.read_sql_query('SELECT * FROM post',    con)
comments = pd.read_sql_query('SELECT * FROM comment', con)
con.close()

# created_utc is unix seconds — convert once so downstream cells get datetimes.
for df in (posts, comments):
    df['created'] = pd.to_datetime(df['created_utc'], unit='s', utc=True)

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 30)

print(f'users:    {len(users):>7,} rows  {users.memory_usage(deep=True).sum()/1e6:6.1f} MB')
print(f'posts:    {len(posts):>7,} rows  {posts.memory_usage(deep=True).sum()/1e6:6.1f} MB')
print(f'comments: {len(comments):>7,} rows  {comments.memory_usage(deep=True).sum()/1e6:6.1f} MB')

users:         56 rows     0.0 MB
posts:    187,920 rows   111.7 MB


comments: 567,990 rows   446.6 MB


## schema

In [2]:
for name, df in [('users', users), ('posts', posts), ('comments', comments)]:
    print(f'\n=== {name} ===')
    print(df.dtypes.to_string())


=== users ===
username              str
author_fullname       str
arctic_meta_json      str
fetched_at          int64

=== posts ===
post_id                           str
author_username                   str
subreddit_name                    str
created_utc                     int64
title                             str
selftext                          str
url                               str
score                           int64
num_comments                    int64
fetched_at                      int64
over_18                         int64
created            datetime64[s, UTC]

=== comments ===
comment_id                        str
author_username                   str
subreddit_name                    str
link_id                           str
parent_id                         str
created_utc                     int64
body                              str
score                           int64
fetched_at                      int64
created            datetime64[s, UTC]


## per-user activity ranking

In [3]:
p_stats = posts.groupby('author_username').agg(posts=('post_id', 'count'),         post_karma=('score', 'sum'))
c_stats = comments.groupby('author_username').agg(comments=('comment_id', 'count'), comment_karma=('score', 'sum'))
rank = p_stats.join(c_stats, how='outer').fillna(0).astype(int)
rank['total'] = rank.posts + rank.comments
rank.sort_values('total', ascending=False).head(20)

,posts,post_karma,comments,comment_karma,total
author_username,,,,,
Kylde,50000,1405972,43958,107499,93958
davidreiss666,50000,2417042,38791,317451,88791
RamsesThePigeon,4583,1134027,50000,4931983,54583
verdatum,386,12354,50000,411849,50386
Apostolate,84,2493,42033,1596513,42117
N8theGr8,6536,2900840,34336,1586619,40872
maxwellhill,37309,39064727,2015,156320,39324
Quouar,6024,711064,30941,261626,36965
LeaderAtLeading,127,507,33078,40203,33205


## NSFW distribution
Only users synced after `over_18` was added will have populated values; older rows default to 0.

In [4]:
nsfw = posts.groupby('author_username').agg(
    total_posts=('post_id', 'count'),
    nsfw_posts=('over_18', 'sum'),
)
nsfw['nsfw_pct'] = (100 * nsfw.nsfw_posts / nsfw.total_posts).round(1)
nsfw[nsfw.nsfw_posts > 0].sort_values('nsfw_pct', ascending=False)

,total_posts,nsfw_posts,nsfw_pct
author_username,,,


## recent activity for `target_user`

In [5]:
user_posts    = posts[posts.author_username == target_user].assign(kind='post',    body=lambda d: d.title.fillna(''))
user_comments = comments[comments.author_username == target_user].assign(kind='comment', body=lambda d: d.body.fillna('').str[:80])
recent = pd.concat([
    user_posts[['created', 'kind', 'subreddit_name', 'score', 'body']],
    user_comments[['created', 'kind', 'subreddit_name', 'score', 'body']],
], ignore_index=True).sort_values('created', ascending=False)
recent.head(25)

,created,kind,subreddit_name,score,body
127,2026-06-04 16:58:59+00:00,comment,ShowYourApp,1,The community feature is probably the most interesting part. Quote apps are ...
128,2026-06-04 16:58:36+00:00,comment,ProductivityGuide,1,"For sensitive notes and documents, yes. The tradeoff is usually convenience ..."
129,2026-06-04 16:58:24+00:00,comment,PromptEngineering,1,The biggest challenge with frameworks like this is moving from interesting t...
130,2026-06-04 16:58:15+00:00,comment,agencynewbies,1,Most agencies get their first clients through people they already know. The ...
131,2026-06-04 16:58:01+00:00,comment,PromptEngineering,1,Pretty much. The difference is persistence and organization. Prompt engineer...
132,2026-06-04 16:57:42+00:00,comment,AIVoice_Agents,1,The interesting part is not the video generation. It is finding products and...
133,2026-06-04 16:57:33+00:00,comment,Businessowners,1,"At 7 figures I would be looking at automation, not spreadsheets. Manually al..."
134,2026-06-04 16:57:18+00:00,comment,ArtificialInteligence,1,That feels like a real problem. Context reuse and memory are probably bigger...
135,2026-06-04 16:57:08+00:00,comment,seogrowth,1,Agreed. Broad keywords maximize traffic. Specific keywords maximize intent. ...
136,2026-06-04 16:56:59+00:00,comment,IndieDev,1,"I think authenticity becomes more valuable, not less. When everyone can gene..."


## sub-by-sub split for `target_user`

In [6]:
up = posts[posts.author_username == target_user].assign(kind='post')
uc = comments[comments.author_username == target_user].assign(kind='comment')
ev = pd.concat([up[['subreddit_name', 'kind', 'score']], uc[['subreddit_name', 'kind', 'score']]])
ev.groupby('subreddit_name').agg(
    posts=('kind', lambda s: (s == 'post').sum()),
    comments=('kind', lambda s: (s == 'comment').sum()),
    karma_sum=('score', 'sum'),
    karma_avg=('score', 'mean'),
).assign(total=lambda d: d.posts + d.comments).sort_values('total', ascending=False).head(25).round(2)

,posts,comments,karma_sum,karma_avg,total
subreddit_name,,,,,
SideProject,14,4386,5567,1.27,4400
micro_saas,24,2260,2822,1.24,2284
buildinpublic,0,1508,1846,1.22,1508
Entrepreneurs,13,1277,1504,1.17,1290
microsaas,17,1200,1453,1.19,1217
DigitalMarketing,7,1116,1536,1.37,1123
saasbuild,7,1109,1387,1.24,1116
SaasDevelopers,0,896,1044,1.17,896
TestMyApp,3,657,723,1.10,660


## top outbound domains for `target_user`
Scans post titles, selftext, post.url, and comment bodies; dedupes URLs per source item.

In [7]:
import re
from collections import Counter

URL_RE = re.compile(r'https?://[^\s\)\]\>"]+')
REDDIT_SUFFIXES = ('reddit.com', 'redd.it', 'redditmedia.com', 'redditstatic.com')

def host(u):
    m = re.match(r'https?://([^/]+)', u)
    return m.group(1).lower() if m else None
def is_reddit(h):
    return any(h.endswith(s) for s in REDDIT_SUFFIXES)

domains = Counter()
for _, p in posts[posts.author_username == target_user].iterrows():
    urls = set(URL_RE.findall(str(p.title or '') + ' ' + str(p.selftext or '')))
    if isinstance(p.url, str):
        urls.add(p.url)
    for h in (host(u) for u in urls):
        if h and not is_reddit(h):
            domains[h] += 1
for body in comments.loc[comments.author_username == target_user, 'body'].dropna():
    for h in (host(u) for u in set(URL_RE.findall(body))):
        if h and not is_reddit(h):
            domains[h] += 1

pd.DataFrame(domains.most_common(20), columns=['domain', 'count'])

,domain,count
0,leadline.dev,2872
1,www.leadline.dev,45
2,leadline.de,2
3,chess.com,2
4,claude.md,2
5,skill.md,1
6,raindrop.io,1
7,fireflies.ai,1
8,moonstruck.ai,1
9,unroll.me,1


## missing-data checks

In [8]:
user_set = set(users.username)
active   = set(posts.author_username) | set(comments.author_username)

print(f'orphan posts    (author not in user): {(~posts.author_username.isin(user_set)).sum()}')
print(f'orphan comments (author not in user): {(~comments.author_username.isin(user_set)).sum()}')
print(f'users with zero activity:             {sorted(user_set - active) or "none"}')

orphan posts    (author not in user): 0
orphan comments (author not in user): 0
users with zero activity:             none


In [9]:
# Per-user date range — useful for spotting the 50K stream cap truncating early history.
all_ev = pd.concat([
    posts[['author_username', 'created']],
    comments[['author_username', 'created']],
])
ranges = all_ev.groupby('author_username').agg(
    events=('created', 'size'),
    first_seen=('created', 'min'),
    last_seen=('created', 'max'),
)
ranges['span_days'] = (ranges.last_seen - ranges.first_seen).dt.days
ranges.sort_values('events', ascending=False).head(15)

,events,first_seen,last_seen,span_days
author_username,,,,
Kylde,93958,2009-07-08 07:01:45+00:00,2026-06-04 08:47:53+00:00,6175
davidreiss666,88791,2006-12-03 00:30:37+00:00,2023-11-24 17:33:36+00:00,6200
RamsesThePigeon,54583,2012-04-18 00:45:27+00:00,2026-05-29 22:41:29+00:00,5154
verdatum,50386,2011-10-05 14:36:00+00:00,2025-09-15 16:03:20+00:00,5094
Apostolate,42117,2012-01-17 15:04:18+00:00,2025-08-14 08:22:14+00:00,4957
N8theGr8,40872,2010-06-18 03:38:12+00:00,2022-08-01 13:12:06+00:00,4427
maxwellhill,39324,2006-03-13 11:54:29+00:00,2020-06-30 21:08:42+00:00,5223
Quouar,36965,2012-01-19 17:09:36+00:00,2026-06-04 09:28:44+00:00,5249
LeaderAtLeading,33205,2026-04-10 11:19:57+00:00,2026-06-04 16:58:59+00:00,55
